# Historical research notebook

Curated from the original collaborative PBL repository (source blob `4c816aa5ae4493ace4141a631e3cfa904d092a14`). Original cell order and research logic are retained; machine-specific PBL paths were replaced by paths relative to this notebook, and saved outputs/execution metadata were cleared for privacy and size. This is an archival notebook, not a verified end-to-end runnable pipeline. Original datasets, embeddings and clustering files are not included. Execute individual cells only after supplying the expected inputs and reviewing alternative branches. Some cells correspond to earlier experiments rather than the final configuration; see [data preparation notes](../../docs/data_preparation.md).


In [ ]:
import sys
sys.path.append('..')

In [ ]:
import pandas as pd
from pymmseqs.commands import easy_cluster
from pathlib import Path
from src.preprocessing import add_enz_flag, truncate, remove_duplicate_seqs, remove_unnatural_seqs, filter_lengths, cluster_df, filter_h5_from_fasta, split_dataset_fasta, undersample, split_and_create_h5
from src.utils import df_to_tsv, df_to_fasta, clear_folder, length_distribution, class_distribution

In [ ]:
df_path = Path('../data/raw/exp_proteins.tsv')
df_raw = pd.read_csv(df_path, sep='\t')

Add enzyme flag and truncate to necessary columns

In [ ]:
df_raw_enz = add_enz_flag(df_raw)
df_raw_enz_trunc = truncate(df_raw_enz, ['Entry', 'Entry Name', 'Sequence', 'Enzyme'])

Remove unnatural sequences entirely before clustering

In [ ]:
df_raw_enz_trunc_nat = remove_unnatural_seqs(df_raw_enz_trunc)

Split dataset into enzymes and non-enzymes

In [ ]:
enz_df = df_raw_enz_trunc_nat[df_raw_enz_trunc_nat['Enzyme'] == 1]
nonenz_df = df_raw_enz_trunc_nat[df_raw_enz_trunc_nat['Enzyme'] == 0]

Clustering

In [ ]:
min_seq_id = 0.3
# First save to temporary fasta file
enz_temp_path = Path('../temp/enz_df_temp.fasta')
nonenz_temp_path = Path('../temp/nonenz_df_temp.fasta')
enz_prefix = Path('../temp/enz_df')
nonenz_prefix = Path('../temp/nonenz_df')

df_to_fasta(enz_df, enz_temp_path)
df_to_fasta(nonenz_df, nonenz_temp_path)

In [ ]:

# Cluster seperately
easy_cluster(fasta_files=enz_temp_path, cluster_prefix=enz_prefix, tmp_dir='../temp/', min_seq_id=min_seq_id)
easy_cluster(fasta_files=nonenz_temp_path, cluster_prefix=nonenz_prefix, tmp_dir='../temp/', min_seq_id=min_seq_id)

In [ ]:
enz_cmap_path = Path(f'{enz_prefix}_cluster.tsv')
nonenz_cmap_path = Path(f'{nonenz_prefix}_cluster.tsv')

In [ ]:
# Get seperately clustered DataFrames and put them together
# In case of exact duplicates, remove them

enz_df_clust = cluster_df(enz_df, enz_cmap_path)
nonenz_df_clust = cluster_df(nonenz_df, nonenz_cmap_path)

df_clust = remove_duplicate_seqs(pd.concat([enz_df_clust, nonenz_df_clust]))

In [ ]:
# Alternatively read in df_clust from existing dataset
df_clust = pd.read_csv('../data/clustered_new/exp_proteins_c30_enz.tsv', sep='\t')

Save clustered df to data/clustered_new/

In [ ]:
df_to_tsv(df_clust, '../data/clustered_new/exp_proteins_c30_enz.tsv')
df_to_fasta(df_clust, '../data/clustered_new/exp_proteins_c30_enz.fasta')

In [ ]:
clear_folder('../temp')

### Further preprocessing

In [ ]:
# For file naming
flens = ''
usamp = ''

Filter lengths

In [ ]:
p1 = 2
p2 = 98
df_final = filter_lengths(df_clust, p1, p2)
flens = f'_flens{p1}-{p2}'

Undersample

In [ ]:
df_final = undersample(df_final)
usamp = '_usamp'

Save preprocessed DataFrame

In [ ]:
out_file = Path(f'../data/preprocessed_new/exp_proteins_c{int(min_seq_id*100)}{flens}{usamp}_enz')
df_to_fasta(df_final, f'{out_file}.fasta')
df_to_tsv(df_final, f'{out_file}.tsv')

Create embeddings file based on dataset

In [ ]:
fasta = Path('../data/preprocessed_new/exp_proteins_c30_flens2-98_usamp_enz.fasta')

In [ ]:
filter_h5_from_fasta('../data/embeddings/embeddings_uniprot_exp_proteins.h5', fasta)

Split dataset

In [ ]:
split_dataset_fasta(fasta, train_size=0.85)

In [ ]:
split_and_create_h5(fasta_path=fasta, train_size=0.85)

Create embedding files for each split

In [ ]:
train_file = f'../data/split/{fasta.stem}_train.fasta'
test_file = f'../data/split/{fasta.stem}_test.fasta'
filter_h5_from_fasta('../data/embeddings/embeddings_uniprot_exp_proteins.h5', train_file)
filter_h5_from_fasta('../data/embeddings/embeddings_uniprot_exp_proteins.h5', test_file)

Use dim reduction on h5 embeddings

In [ ]:
from src.preprocessing import dim_reduction_train_test, create_dataset_from_h5

train_set = create_dataset_from_h5('../data/embeddings/embeddings_uniprot_exp_proteins_c30_flens2-98_enz/embeddings_uniprot_exp_proteins_c30_flens2-98_enz_train.h5')
test_set = create_dataset_from_h5('../data/embeddings/embeddings_uniprot_exp_proteins_c30_flens2-98_enz/embeddings_uniprot_exp_proteins_c30_flens2-98_enz_test.h5')

dim_reduction_train_test(train_set=train_set, test_set=test_set, n_dims=128, out_path='../data/embeddings/embeddings_uniprot_exp_proteins_c30_flens2-98_enz/')